[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Row Factories &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the stations and their year of readings, in the two
tables the notebook used. Run it first. The tasks do not depend on one another, and the last cell
removes the scratch folder.


In [1]:
import json
import math
import shutil
import sqlite3
from contextlib import closing
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


**1.** Tromso's coldest reading, by name.


In [2]:
with closing(sqlite3.connect(DATABASE)) as conn:
    conn.row_factory = sqlite3.Row
    row = conn.execute("""
        SELECT r.hour, r.celsius
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE s.name = ? AND r.celsius IS NOT NULL
        ORDER BY r.celsius, r.hour
        LIMIT 1
    """, ("Tromso",)).fetchone()

print(row["celsius"], "at", row["hour"])


-9.2 at 2025-01-08T03:00


The factory has to be set before the query runs, since `conn.execute` gives the new cursor whatever
the connection's factory is at that moment. The row keeps its values after the connection closes, so
reading it outside the block works. The column `r.hour` is named `hour` in the row, without its
table.


**2.** Names in the query's own case.


In [3]:
with closing(sqlite3.connect(DATABASE)) as conn:
    conn.row_factory = sqlite3.Row
    row = conn.execute("SELECT name AS Station, latitude AS Lat FROM stations ORDER BY latitude DESC").fetchone()

print(row.keys())
print(row["lat"])


['Station', 'Lat']
78.22


`keys()` reports the names exactly as the query wrote them, capitals and all, while a lookup ignores
case, so `row["lat"]` found `Lat`. A dictionary made with `dict(row)` would keep `Station` and `Lat`,
and answer only to those.


**3.** Any query, as JSON.


In [4]:
def dict_factory(cursor, row):
    """A row as a dictionary, keyed by the names the query gave its columns."""
    return {column[0]: value for column, value in zip(cursor.description, row)}


def rows_as_json(conn, sql, parameters=()):
    """The rows of a query as a JSON string, fetched as dictionaries on a cursor of their own."""
    cursor = conn.cursor()
    cursor.row_factory = dict_factory
    return json.dumps(cursor.execute(sql, parameters).fetchall())


with closing(sqlite3.connect(DATABASE)) as conn:
    print(rows_as_json(conn, "SELECT name, latitude FROM stations WHERE latitude > ? ORDER BY latitude DESC", (69,)))


[{"name": "Svalbard", "latitude": 78.22}, {"name": "Kirkenes", "latitude": 69.73}, {"name": "Tromso", "latitude": 69.65}]


The factory is set on the function's own cursor, so a caller's connection keeps whatever factory it
had. The dictionaries go to `json.dumps` as they are, since a list of dictionaries of strings and
numbers is exactly what JSON writes.


**4.** Readings as dataclasses.


In [5]:
@dataclass
class Reading:
    station: str
    hour: str
    celsius: float | None


def reading_factory(cursor, row):
    """A row as a Reading, with every column matched to the field of the same name."""
    return Reading(**{column[0]: value for column, value in zip(cursor.description, row)})


with closing(sqlite3.connect(DATABASE)) as conn:
    cursor = conn.cursor()
    cursor.row_factory = reading_factory
    warmest = cursor.execute("""
        SELECT s.name AS station, r.hour, r.celsius
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE s.name = ? AND r.celsius IS NOT NULL
        ORDER BY r.celsius DESC, r.hour
        LIMIT 3
    """, ("Oslo",)).fetchall()

for reading in warmest:
    print(reading)


Reading(station='Oslo', hour='2025-07-17T15:00', celsius=19.3)
Reading(station='Oslo', hour='2025-07-21T15:00', celsius=19.2)
Reading(station='Oslo', hour='2025-07-22T14:00', celsius=19.2)


The query names its columns to match the fields: `s.name AS station`, and `r.hour` and `r.celsius`,
which arrive as `hour` and `celsius`. A column with no matching field would make the factory raise
`TypeError` at the first row.


**5.** A plain list of names.


In [6]:
def first_value(cursor, row):
    """Only the first value of a row."""
    return row[0]


with closing(sqlite3.connect(DATABASE)) as conn:
    cursor = conn.cursor()
    cursor.row_factory = first_value
    names = cursor.execute("SELECT name FROM stations ORDER BY name").fetchall()

print(names)


['Bergen', 'Kirkenes', 'Oslo', 'Svalbard', 'Tromso']


A factory can return anything, including a single value, so `fetchall` gave a list of strings
instead of a list of one-value tuples, with no unpacking afterward.


**6.** Stations to a JSON file, and back.


In [7]:
with closing(sqlite3.connect(DATABASE)) as conn:
    conn.row_factory = sqlite3.Row
    stations = [dict(row) for row in conn.execute("SELECT id, name, latitude FROM stations ORDER BY id")]

path = SCRATCH / "stations.json"
with path.open("w") as file:
    json.dump(stations, file, indent=2)
with path.open() as file:
    loaded = json.load(file)

print([station["name"] for station in loaded])


['Bergen', 'Oslo', 'Svalbard', 'Tromso', 'Kirkenes']


`dict(row)` turned every `Row` into something `json.dump` can write, and `json.load` gave back a
list of dictionaries, with the same names as the query's columns.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Row Factories](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/05-row-factories.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
